# Modelo con LTSM, este modelo ya incorpora teching force

In [1]:
version = "5_1"

In [2]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [3]:
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)
    
with open("../../data/bicycles/df_bicycles_stations.pk1", "rb") as f:
    df_bikesStations = pickle.load(f)

In [ ]:
df_model = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual"
])

In [5]:
# Variables de entrada
context = df_model.drop(columns=['start_station_idx'])
start = df_model['start_station_idx']
end   = df_model['end_station_idx']

In [6]:
context.shape

(9510782, 28)

In [7]:
start.shape

(9510782,)

In [8]:
end.shape

(9510782,)

In [11]:
from sklearn.model_selection import train_test_split

ctx_train, ctx__test, start_train, start_test, end_train, end_test = train_test_split(
    context, 
    start, 
    end, 
    test_size=0.2, 
    random_state=42
)

# ---------- Normalizar índices de estaciones con un offset común
station_offset = int(min(start.min(), end.min()))
# desplazamos todos para que el mínimo sea 0
# start_train_shift = (start_train - station_offset).astype(int)
# start_test_shift  = (start_test  - station_offset).astype(int)
# end_train_shift   = (end_train   - station_offset).astype(int)
# end_test_shift    = (end_test    - station_offset).astype(int)

num_stations = int(max(start.max(), end.max()) - station_offset + 1)
num_features = context.shape[1]

print(f"Number of stations: {num_stations}")
print(f"Number of features: {num_features}")
print(f"Station offset: {station_offset}")

# ---------- Preparar inputs en la forma que requieren los Embeddings: (n,1)
input_train_context = ctx_train.values.astype(np.float32)
input_test_context  = ctx__test.values.astype(np.float32)

input_train_start = start_train.values.reshape(-1, 1)
input_test_start  = start_test.values.reshape(-1, 1)

# Labels (etiquetas) para clasificación: estación destino (y_end)
output_train_end = end_train.values.astype(int)
output_test_end  =end_test.values.astype(int)

# ---------- split del train en train/val
input_train_context, input_val_context, input_train_start, input_val_start, output_train_end, output_val_end = train_test_split(
    input_train_context,
    input_train_start,
    output_train_end,
    test_size=0.2,
    random_state=42,
    shuffle=True # No se puede estratificar porque hay estaciones con muy pocos datos
)

Number of stations: 1912
Number of features: 28
Station offset: 0


In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

embedding_dim = int(np.ceil(np.sqrt(num_stations)))
timesteps = 1  # ya que tu contexto no es secuencia real

print(f"Embedding dimension: {embedding_dim}")

# ---------------------------
# Input 1: contexto (vector)
# ---------------------------
context_input = layers.Input(shape=(num_features,), name="context")

# Convertimos el vector en una "secuencia" de longitud 1
x_ctx = layers.Reshape((1, num_features))(context_input)

# LSTM procesando el contexto
x_ctx = layers.LSTM(128, return_sequences=False)(x_ctx)
x_ctx = layers.Dropout(0.3)(x_ctx)

# ---------------------------
# Input 2: estación origen
# ---------------------------
start_input = layers.Input(shape=(1,), name="start_station_input")

start_emb = layers.Embedding(num_stations, embedding_dim)(start_input)
start_emb = layers.Flatten()(start_emb)
start_emb = layers.Dense(32, activation="relu")(start_emb)

# ---------------------------
# Concatenación
# ---------------------------

x = layers.Concatenate()([x_ctx, start_emb])

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation="relu")(x)

end_output = layers.Dense(num_stations, activation="softmax")(x)

model = Model(inputs=[context_input, start_input], outputs=end_output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Embedding dimension: 44


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ context             │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_station_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 28)     │          0 │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 44)     │     84,128 │ start_station_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │     80,384 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 44)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,440 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     20,608 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1912)      │    124,280 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 319,096 (1.22 MB)

 Trainable params: 319,096 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    {"context": input_train_context, "start_station_input": input_train_start},
    output_train_end,
    validation_data=(
        {"context": input_val_context, "start_station_input": input_val_start},
        output_val_end
    ),
    epochs=50,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)


Epoch 1/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 365s 15ms/step - accuracy: 0.2189 - loss: 3.0142 - val_accuracy: 0.2635 - val_loss: 2.8230
Epoch 2/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 374s 16ms/step - accuracy: 0.2715 - loss: 2.7344 - val_accuracy: 0.1897 - val_loss: 3.6575
Epoch 3/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 350s 15ms/step - accuracy: 0.2888 - loss: 2.6585 - val_accuracy: 0.1726 - val_loss: 4.4049
Epoch 4/50
10226/23777 ━━━━━━━━━━━━━━━━━━━━ 2:54 13ms/step - accuracy: 0.2883 - loss: 2.6849

KeyboardInterrupt: 